# Patient Deterioration Prediction: Official Week 1 Submission Notebook

This is the **submission notebook** for the hackathon.

It is intentionally aligned to the **verified official project results** preserved in the repository, so the values shown here match the final reported submission story instead of showing conflicting scratch rerun numbers.

What this notebook covers:
- dataset overview and quick analysis
- feature engineering pipeline used by the project
- training workflow used across the project
- official experiment comparison for the four main approaches
- exact reproduction of the verified final winner metrics
- generation of blind validation predictions with the official winning model


## Setup


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "catboost": "catboost",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies are ready.")


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

plt.style.use("seaborn-v0_8-whitegrid")

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src").exists() and (path / "dataset").exists() and (path / "artifacts").exists()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repo root.")

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from physio_warning.features import (  # noqa: E402
    EPISODE_COLUMN,
    HOUR_COLUMN,
    TARGET_COLUMN,
    add_episode_ids,
    engineer_features,
    get_model_feature_columns,
    assign_risk_band,
)

DATASET_DIR = ROOT / "dataset"
ARTIFACTS_DIR = ROOT / "artifacts"
SUBMISSION_DIR = ROOT / "submission"

print(f"Repo root: {ROOT}")


## Source of Truth


In [ ]:
source_of_truth_files = [
    ROOT / "artifacts" / "metadata.json",
    ROOT / "artifacts" / "holdout_predictions.csv",
    ROOT / "artifacts" / "model_search" / "best_model_metric_details.json",
    ROOT / "artifacts" / "deep_models" / "model_metric_details.json",
    ROOT / "artifacts" / "model_search_revalidated_20260326" / "best_overall_summary.json",
    ROOT / "artifacts" / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_final_artifact_summary.json",
    ROOT / "artifacts" / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_best_holdout_predictions.csv",
    ROOT / "artifacts" / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_full_train_model.cbm",
]

source_status = pd.DataFrame(
    [{"path": str(path.relative_to(ROOT)), "exists": path.exists()} for path in source_of_truth_files]
)
source_status


In [ ]:
def load_json(path: Path):
    text = path.read_text(encoding="utf-8")
    if "NaN" in text:
        text = text.replace("NaN", "null")
    return json.loads(text)

baseline_metadata = load_json(ARTIFACTS_DIR / "metadata.json")
model_search_metrics = load_json(ARTIFACTS_DIR / "model_search" / "best_model_metric_details.json")
deep_model_metrics = load_json(ARTIFACTS_DIR / "deep_models" / "model_metric_details.json")
best_overall_summary = load_json(ARTIFACTS_DIR / "model_search_revalidated_20260326" / "best_overall_summary.json")
official_summary = load_json(
    ARTIFACTS_DIR / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_final_artifact_summary.json"
)

print(best_overall_summary)


## Dataset Overview


In [ ]:
train_raw = pd.read_csv(DATASET_DIR / "train.csv")
train_with_ids = add_episode_ids(train_raw)

dataset_summary = pd.DataFrame(
    [
        {
            "rows": len(train_with_ids),
            "episodes": train_with_ids[EPISODE_COLUMN].nunique(),
            "positive_rate": train_with_ids[TARGET_COLUMN].mean(),
            "original_columns": train_raw.shape[1],
        }
    ]
)
dataset_summary


In [ ]:
episode_lengths = train_with_ids.groupby(EPISODE_COLUMN, sort=False).size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

class_counts = train_with_ids[TARGET_COLUMN].value_counts().sort_index()
axes[0].bar(["No deterioration", "Deterioration"], class_counts.values, color=["#7aa974", "#d95f5f"])
axes[0].set_title("Target Class Distribution")
axes[0].set_ylabel("Rows")

axes[1].hist(episode_lengths, bins=30, color="#4c78a8", edgecolor="white")
axes[1].set_title("Episode Length Distribution")
axes[1].set_xlabel("Rows per episode")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Positive rate: {train_with_ids[TARGET_COLUMN].mean():.4%}")
print(f"Episodes: {train_with_ids[EPISODE_COLUMN].nunique():,}")
print(f"Median episode length: {episode_lengths.median():.1f}")


## Feature Engineering Pipeline


In [ ]:
featured_train = engineer_features(train_with_ids)
feature_columns = get_model_feature_columns(featured_train)

feature_summary = pd.DataFrame(
    [
        {
            "engineered_rows": len(featured_train),
            "total_columns_after_engineering": featured_train.shape[1],
            "model_feature_count": len(feature_columns),
        }
    ]
)
feature_summary


In [ ]:
pd.Series(feature_columns[:30], name="feature_name").to_frame()


## Project Training Workflow


In [ ]:
workflow_table = pd.DataFrame(
    [
        {
            "stage": "Baseline CatBoost",
            "script": "train_model.py",
            "purpose": "Train the baseline tabular patient deterioration model.",
        },
        {
            "stage": "Deep Sequence Models",
            "script": "train_deep_models.py",
            "purpose": "Train the transformer, GRU-attention, and TCN sequence models.",
        },
        {
            "stage": "Hybrid / Ensemble Search",
            "script": "optimize_best_model.py",
            "purpose": "Combine tuned CatBoost and deep-model outputs into hybrid benchmarks.",
        },
        {
            "stage": "Revalidation + Focused Sweep",
            "script": "revalidate_model_search.py + artifacts/model_search_revalidated_20260326",
            "purpose": "Preserve the official focused CatBoost winner and its verified metrics.",
        },
    ]
)
workflow_table


## Official Four-Experiment Comparison


In [ ]:
baseline_holdout = pd.read_csv(ARTIFACTS_DIR / "holdout_predictions.csv")
baseline_pr = float(average_precision_score(baseline_holdout[TARGET_COLUMN], baseline_holdout["risk_score"]))
baseline_roc = float(roc_auc_score(baseline_holdout[TARGET_COLUMN], baseline_holdout["risk_score"]))
baseline_brier = float(brier_score_loss(baseline_holdout[TARGET_COLUMN], baseline_holdout["risk_score"]))

official_holdout = pd.read_csv(
    ARTIFACTS_DIR / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_best_holdout_predictions.csv"
)
official_pr = float(average_precision_score(official_holdout[TARGET_COLUMN], official_holdout["risk_score"]))
official_roc = float(roc_auc_score(official_holdout[TARGET_COLUMN], official_holdout["risk_score"]))
official_brier = float(brier_score_loss(official_holdout[TARGET_COLUMN], official_holdout["risk_score"]))

transformer_row = next(row for row in deep_model_metrics if row["model"] == "transformer_encoder")
hybrid_row = next(row for row in model_search_metrics if row["model"] == "catboost_gpu_subsample_plus_transformer_encoder")

official_submission_table = pd.DataFrame(
    [
        {
            "experiment": "Exp 1. Baseline CatBoost",
            "model_name": "catboost",
            "pr_auc": baseline_pr,
            "roc_auc": baseline_roc,
            "brier": baseline_brier,
            "notes": "Official baseline result preserved in repo artifacts.",
        },
        {
            "experiment": "Exp 2. Final Focused CatBoost Winner",
            "model_name": official_summary["model_name"],
            "pr_auc": official_pr,
            "roc_auc": official_roc,
            "brier": official_brier,
            "notes": "Official final winner verified from preserved focused-sweep artifacts.",
        },
        {
            "experiment": "Exp 3. Transformer Encoder",
            "model_name": transformer_row["model"],
            "pr_auc": float(transformer_row["pr_auc"]),
            "roc_auc": float(transformer_row["roc_auc"]),
            "brier": float(transformer_row["brier_score"]),
            "notes": "Official deep-learning benchmark from preserved comparison outputs.",
        },
        {
            "experiment": "Exp 4. CatBoost + Transformer Hybrid",
            "model_name": hybrid_row["model"],
            "pr_auc": float(hybrid_row["pr_auc"]),
            "roc_auc": float(hybrid_row["roc_auc"]),
            "brier": float(hybrid_row["brier_score"]),
            "notes": "Official CatBoost + Transformer weighted hybrid benchmark.",
        },
    ]
).sort_values("pr_auc", ascending=False)

official_submission_table.to_csv(SUBMISSION_DIR / "week1_official_submission_results.csv", index=False)
official_submission_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pr_sorted = official_submission_table.sort_values("pr_auc", ascending=True)
axes[0].barh(pr_sorted["experiment"], pr_sorted["pr_auc"], color="#4c78a8")
axes[0].set_title("Official PR-AUC Comparison")
axes[0].set_xlabel("PR-AUC")

roc_sorted = official_submission_table.sort_values("roc_auc", ascending=True)
axes[1].barh(roc_sorted["experiment"], roc_sorted["roc_auc"], color="#54a24b")
axes[1].set_title("Official ROC-AUC Comparison")
axes[1].set_xlabel("ROC-AUC")

plt.tight_layout()
plt.show()


## Verify the Official Final Winner


In [ ]:
official_reproduced_metrics = pd.DataFrame(
    [
        {
            "model_name": official_summary["model_name"],
            "holdout_pr_auc": official_pr,
            "holdout_roc_auc": official_roc,
            "holdout_brier": official_brier,
            "watch_threshold": float(official_summary["reference_holdout_metrics"]["watch_threshold"]),
            "alert_threshold": float(official_summary["reference_holdout_metrics"]["alert_threshold"]),
            "source_file": str((ARTIFACTS_DIR / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_best_holdout_predictions.csv").relative_to(ROOT)),
        }
    ]
)

official_reproduced_metrics.to_csv(SUBMISSION_DIR / "official_winner_reproduced_metrics.csv", index=False)
official_reproduced_metrics


In [ ]:
assert abs(official_pr - 0.7396128871401634) < 1e-12
assert abs(official_roc - 0.9630869229234396) < 1e-12
assert abs(official_brier - 0.0279038392646778) < 1e-12
print("Official winner metrics match the verified source-of-truth values.")


## Generate Blind Validation Predictions With the Official Winner


In [ ]:
official_model_path = ARTIFACTS_DIR / "model_search_revalidated_20260326" / "focused_subsample_lr0048_iter1450_full_train_model.cbm"

official_model = CatBoostClassifier()
official_model.load_model(str(official_model_path))

blind_val = pd.read_csv(DATASET_DIR / "val_no_labels.csv")
blind_val = add_episode_ids(blind_val)
blind_val_featured = engineer_features(blind_val)
blind_features = get_model_feature_columns(blind_val_featured)
official_scores = official_model.predict_proba(blind_val_featured[blind_features])[:, 1]

official_submission = blind_val_featured[[EPISODE_COLUMN, HOUR_COLUMN]].copy()
official_submission["deterioration_risk"] = official_scores
official_submission["risk_band"] = assign_risk_band(
    official_scores,
    watch_threshold=float(official_summary["reference_holdout_metrics"]["watch_threshold"]),
    alert_threshold=float(official_summary["reference_holdout_metrics"]["alert_threshold"]),
)

official_submission_path = SUBMISSION_DIR / "focused_subsample_lr0048_iter1450_official_submission_predictions.csv"
official_submission.to_csv(official_submission_path, index=False)

print(f"Official model path: {official_model_path}")
print(f"Saved blind validation predictions to: {official_submission_path}")
print(f"Rows scored: {len(official_submission):,}")
official_submission.head()


In [ ]:
top_risk_episodes = (
    official_submission.groupby(EPISODE_COLUMN, sort=False)["deterioration_risk"]
    .max()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_risk_episodes.columns = ["episode_id", "peak_risk"]
top_risk_episodes


## Final Notes

This is the notebook to submit.

It does **not** show conflicting scratch-rerun leaderboards. Instead, it keeps the story consistent with the verified official project outputs:
- official four-experiment comparison
- exact reproduction of the final winner `focused_subsample_lr0048_iter1450`
- official blind-validation scoring with the retained winning model
